# Notebook 17: fixed-particle displacement pilot

This notebook performs a bounded pilot study of a fixed-particle displacement representation for the particle-first route.

The pilot uses only the validated compact cache for size ratio $1.000000$, simulation 1. It does not access raw DEM files and does not perform a full parameter sweep.

For particle $i$, the displacement representation is

$$
d_i(t)=x_i(t)-x_i(t_{\mathrm{ref}}).
$$

The reference snapshot is selected from the training data only.

The absolute-coordinate and displacement-coordinate representations are compared using fixed POD ranks 27 and 100. Diagnostics include singular-value decay, particle reconstruction error, coarse-grained field error, physical validity, and mixing/segregation measures.

This is an exploratory mechanism diagnostic. It does not replace the main Notebook 12 or Notebook 16 route-order conclusions, and it does not claim to reproduce the full intrusive particle solver.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

while (
    not (PROJECT_ROOT / "notebooks").is_dir()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "notebooks"
).is_dir(), PROJECT_ROOT

assert (
    PROJECT_ROOT / "data" / "interim"
).is_dir(), PROJECT_ROOT


N17_CASE_ID = "ratio_1_000000_num_1"

N17_CACHE_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "particle_positions_ratio_1_000000_num_1.npz"
)

N17_EXPECTED_CACHE_SHA256 = (
    "97bddffd33e2bcf8f6863cf29d5f82f5268c9a225bdac3f4399ca4704db3a9a1"
)

N17_RANKS = (27, 100)
N17_GRID_SIZE = 100
N17_CUTOFF = 3.0
N17_TIME_MAX = 1720.72

N17_EXPECTED_TRAINING_COUNT = 101
N17_EXPECTED_VALIDATION_COUNT = 99

N17_TABLE_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "comoving_particle_pilot"
)

N17_FIGURE_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "comoving_particle_pilot"
)


def n17_sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert N17_CACHE_PATH.is_file()

n17_observed_cache_sha256 = (
    n17_sha256_file(N17_CACHE_PATH)
)

assert (
    n17_observed_cache_sha256
    == N17_EXPECTED_CACHE_SHA256
)

print("Notebook 17 preflight passed:", True)
print("Repository root:", PROJECT_ROOT)
print("Case:", N17_CASE_ID)
print("Cache:", N17_CACHE_PATH)
print("Cache SHA-256:", n17_observed_cache_sha256)
print("Ranks:", N17_RANKS)
print(
    "Grid and cutoff:",
    N17_GRID_SIZE,
    N17_CUTOFF,
)
print("Files written:", False)
print("Raw DEM files accessed:", False)

Notebook 17 preflight passed: True
Repository root: /Users/rallen/Documents/msc-rom-particle-systems
Case: ratio_1_000000_num_1
Cache: /Users/rallen/Documents/msc-rom-particle-systems/data/interim/particle_positions_ratio_1_000000_num_1.npz
Cache SHA-256: 97bddffd33e2bcf8f6863cf29d5f82f5268c9a225bdac3f4399ca4704db3a9a1
Ranks: (27, 100)
Grid and cutoff: 100 3.0
Files written: False
Raw DEM files accessed: False


In [2]:
with np.load(
    N17_CACHE_PATH,
    allow_pickle=False,
) as n17_cache:

    n17_cache_keys = sorted(
        n17_cache.files
    )

    positions_xyz = n17_cache[
        "positions_xyz"
    ]

    particle_radii = n17_cache[
        "particle_radii"
    ]

    species_index = n17_cache[
        "species_index"
    ]

    snapshot_times = n17_cache[
        "snapshot_times"
    ]

    source_snapshot_numbers = n17_cache[
        "source_snapshot_numbers"
    ]

    cycle_labels = n17_cache[
        "cycle_labels"
    ]


assert positions_xyz.shape == (
    201,
    10792,
    3,
)

assert particle_radii.shape == (
    10792,
)

assert species_index.shape == (
    10792,
)

assert snapshot_times.shape == (
    201,
)

assert source_snapshot_numbers.shape == (
    201,
)

assert cycle_labels.shape == (
    201,
)

assert np.isfinite(
    positions_xyz
).all()

assert np.isfinite(
    particle_radii
).all()

assert np.isfinite(
    snapshot_times
).all()

assert np.all(
    np.diff(snapshot_times) > 0
)

n17_small_particle_mask = (
    species_index == 1
)

n17_small_particle_count = int(
    n17_small_particle_mask.sum()
)

assert n17_small_particle_count == 5396

print("Cache keys:", n17_cache_keys)
print("Positions shape:", positions_xyz.shape)
print(
    "Small-particle count:",
    n17_small_particle_count,
)
print(
    "Time range:",
    snapshot_times[[0, -1]],
)
print("Unique cycle labels:")
print(
    pd.Series(cycle_labels)
    .value_counts()
    .sort_index()
)
print("Files written:", False)
print("Raw DEM files accessed:", False)

Cache keys: ['cycle_labels', 'particle_radii', 'positions_xyz', 'snapshot_times', 'source_byte_offsets', 'source_snapshot_numbers', 'species_index']
Positions shape: (201, 10792, 3)
Small-particle count: 5396
Time range: [1378.133363 1720.721256]
Unique cycle labels:
cycle_1    101
cycle_2    100
Name: count, dtype: int64
Files written: False
Raw DEM files accessed: False


In [3]:
N17_TIME_ATOL = 1e-8

n17_comparison_mask = (
    snapshot_times
    <= N17_TIME_MAX + N17_TIME_ATOL
)

n17_training_mask = (
    (cycle_labels == "cycle_1")
    & n17_comparison_mask
)

n17_validation_mask = (
    (cycle_labels == "cycle_2")
    & n17_comparison_mask
)

n17_excluded_mask = (
    ~n17_comparison_mask
)

n17_training_indices = np.flatnonzero(
    n17_training_mask
)

n17_validation_indices = np.flatnonzero(
    n17_validation_mask
)

n17_excluded_indices = np.flatnonzero(
    n17_excluded_mask
)

assert len(n17_training_indices) == (
    N17_EXPECTED_TRAINING_COUNT
)

assert len(n17_validation_indices) == (
    N17_EXPECTED_VALIDATION_COUNT
)

assert len(n17_excluded_indices) == 1

assert not np.intersect1d(
    n17_training_indices,
    n17_validation_indices,
).size

assert not np.intersect1d(
    n17_training_indices,
    n17_excluded_indices,
).size

assert not np.intersect1d(
    n17_validation_indices,
    n17_excluded_indices,
).size


# The reference is the first retained training snapshot.
n17_reference_snapshot_index = int(
    n17_training_indices[0]
)

n17_reference_time = float(
    snapshot_times[
        n17_reference_snapshot_index
    ]
)

n17_reference_positions = (
    positions_xyz[
        n17_reference_snapshot_index
    ].copy()
)


n17_absolute_training_matrix = (
    positions_xyz[
        n17_training_indices
    ]
    .reshape(
        len(n17_training_indices),
        -1,
    )
    .T
)

n17_absolute_validation_matrix = (
    positions_xyz[
        n17_validation_indices
    ]
    .reshape(
        len(n17_validation_indices),
        -1,
    )
    .T
)

n17_displacement_training_matrix = (
    positions_xyz[
        n17_training_indices
    ]
    - n17_reference_positions[
        None,
        :,
        :,
    ]
).reshape(
    len(n17_training_indices),
    -1,
).T

n17_displacement_validation_matrix = (
    positions_xyz[
        n17_validation_indices
    ]
    - n17_reference_positions[
        None,
        :,
        :,
    ]
).reshape(
    len(n17_validation_indices),
    -1,
).T


assert n17_absolute_training_matrix.shape == (
    10792 * 3,
    101,
)

assert n17_absolute_validation_matrix.shape == (
    10792 * 3,
    99,
)

assert (
    n17_displacement_training_matrix.shape
    == n17_absolute_training_matrix.shape
)

assert (
    n17_displacement_validation_matrix.shape
    == n17_absolute_validation_matrix.shape
)

assert np.isfinite(
    n17_absolute_training_matrix
).all()

assert np.isfinite(
    n17_absolute_validation_matrix
).all()

assert np.isfinite(
    n17_displacement_training_matrix
).all()

assert np.isfinite(
    n17_displacement_validation_matrix
).all()

print(
    "Training snapshots:",
    len(n17_training_indices),
)

print(
    "Validation snapshots:",
    len(n17_validation_indices),
)

print(
    "Excluded snapshot indices:",
    n17_excluded_indices,
)

print(
    "Excluded time:",
    snapshot_times[
        n17_excluded_indices
    ],
)

print(
    "Reference snapshot index:",
    n17_reference_snapshot_index,
)

print(
    "Reference time:",
    n17_reference_time,
)

print(
    "Absolute matrix shape:",
    n17_absolute_training_matrix.shape,
)

print(
    "Displacement matrix shape:",
    n17_displacement_training_matrix.shape,
)

print("Training-only reference verified:", True)
print("Files written:", False)
print("Raw DEM files accessed:", False)

Training snapshots: 101
Validation snapshots: 99
Excluded snapshot indices: [200]
Excluded time: [1720.721256]
Reference snapshot index: 0
Reference time: 1378.133363
Absolute matrix shape: (32376, 101)
Displacement matrix shape: (32376, 101)
Training-only reference verified: True
Files written: False
Raw DEM files accessed: False


In [4]:
def n17_thin_pod(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    assert matrix.ndim == 2
    assert np.isfinite(matrix).all()

    modes, singular_values, right_vectors = (
        np.linalg.svd(
            matrix,
            full_matrices=False,
        )
    )

    squared_singular_values = (
        singular_values**2
    )

    total_energy = float(
        squared_singular_values.sum()
    )

    assert total_energy > 0.0

    cumulative_energy = (
        np.cumsum(
            squared_singular_values
        )
        / total_energy
    )

    numerical_rank = int(
        np.linalg.matrix_rank(matrix)
    )

    orthonormality_residual = float(
        np.max(
            np.abs(
                modes.T
                @ modes
                - np.eye(modes.shape[1])
            )
        )
    )

    assert np.isfinite(
        singular_values
    ).all()

    assert np.isfinite(
        cumulative_energy
    ).all()

    assert orthonormality_residual < 1e-12

    return {
        "modes": modes,
        "singular_values": singular_values,
        "right_vectors": right_vectors,
        "cumulative_energy": cumulative_energy,
        "numerical_rank": numerical_rank,
        "orthonormality_residual": (
            orthonormality_residual
        ),
    }


# Match Notebook 12's training-centred convention.
n17_absolute_training_mean = (
    n17_absolute_training_matrix.mean(
        axis=1,
        keepdims=True,
    )
)

n17_displacement_training_mean = (
    n17_displacement_training_matrix.mean(
        axis=1,
        keepdims=True,
    )
)

n17_absolute_training_centered = (
    n17_absolute_training_matrix
    - n17_absolute_training_mean
)

n17_absolute_validation_centered = (
    n17_absolute_validation_matrix
    - n17_absolute_training_mean
)

n17_displacement_training_centered = (
    n17_displacement_training_matrix
    - n17_displacement_training_mean
)

n17_displacement_validation_centered = (
    n17_displacement_validation_matrix
    - n17_displacement_training_mean
)


n17_absolute_pod = n17_thin_pod(
    n17_absolute_training_centered
)

n17_displacement_pod = n17_thin_pod(
    n17_displacement_training_centered
)


assert (
    n17_absolute_pod["numerical_rank"]
    >= max(N17_RANKS)
)

assert (
    n17_displacement_pod["numerical_rank"]
    >= max(N17_RANKS)
)

print(
    "Absolute-coordinate numerical rank:",
    n17_absolute_pod[
        "numerical_rank"
    ],
)

print(
    "Displacement-coordinate numerical rank:",
    n17_displacement_pod[
        "numerical_rank"
    ],
)

print(
    "Absolute POD orthonormality residual:",
    n17_absolute_pod[
        "orthonormality_residual"
    ],
)

print(
    "Displacement POD orthonormality residual:",
    n17_displacement_pod[
        "orthonormality_residual"
    ],
)

print(
    "Absolute cumulative energy:",
    {
        rank: float(
            n17_absolute_pod[
                "cumulative_energy"
            ][rank - 1]
        )
        for rank in N17_RANKS
    },
)

print(
    "Displacement cumulative energy:",
    {
        rank: float(
            n17_displacement_pod[
                "cumulative_energy"
            ][rank - 1]
        )
        for rank in N17_RANKS
    },
)

print(
    "POD decomposition completed:",
    True,
)

print("Files written:", False)
print("Raw DEM files accessed:", False)

Absolute-coordinate numerical rank: 100
Displacement-coordinate numerical rank: 100
Absolute POD orthonormality residual: 2.220446049250313e-15
Displacement POD orthonormality residual: 2.6645352591003757e-15
Absolute cumulative energy: {27: 0.9993949216931346, 100: 0.9999999999999998}
Displacement cumulative energy: {27: 0.9993949216931346, 100: 0.9999999999999998}
POD decomposition completed: True
Files written: False
Raw DEM files accessed: False


In [5]:
n17_centered_equivalence_residual = float(
    np.max(
        np.abs(
            n17_absolute_training_centered
            - n17_displacement_training_centered
        )
    )
)

n17_validation_equivalence_residual = float(
    np.max(
        np.abs(
            n17_absolute_validation_centered
            - n17_displacement_validation_centered
        )
    )
)

assert n17_centered_equivalence_residual < 1e-12
assert n17_validation_equivalence_residual < 1e-12

print(
    "Training centred-data equivalence residual:",
    n17_centered_equivalence_residual,
)

print(
    "Validation centred-data equivalence residual:",
    n17_validation_equivalence_residual,
)

print(
    "Fixed-reference displacement is algebraically "
    "equivalent under centred POD:",
    True,
)

print("Files written:", False)

Training centred-data equivalence residual: 3.197442310920451e-14
Validation centred-data equivalence residual: 3.197442310920451e-14
Fixed-reference displacement is algebraically equivalent under centred POD: True
Files written: False


## Fixed-reference displacement equivalence

The fixed-reference displacement representation was tested using

$$
d_t=x_t-x_{\mathrm{ref}},
$$

where $x_{\mathrm{ref}}$ was taken from the training data.

However, the existing particle POD applies training-mean centring. The displacement mean is

$$
\overline{d}
=
\overline{x}-x_{\mathrm{ref}},
$$

so the centred displacement snapshots satisfy

$$
d_t-\overline{d}
=
(x_t-x_{\mathrm{ref}})
-
(\overline{x}-x_{\mathrm{ref}})
=
x_t-\overline{x}.
$$

Therefore, subtracting a fixed reference configuration before applying the existing training-mean centring does not change the centred snapshot matrix. The absolute-coordinate and fixed-reference displacement PODs are algebraically identical, apart from floating-point round-off.

The measured training and validation equivalence residuals were both approximately

$$
3.2\times10^{-14},
$$

well below the prescribed tolerance of $10^{-12}$.

Consequently, this pilot does not provide a distinct alternative particle representation and no separate reconstruction-error or coarse-grained-field results are generated from it.

A genuinely different co-moving study would require a time-dependent translation or rigid-body transformation, or a velocity/state representation. Such an extension would require a separate physical convention and is outside the scope of this bounded pilot.